# **La creation du fichier ventes dirty CSV**


In [18]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [19]:
N_ROWS= 200 
SEED= 42
OUTPUT_PATH = f"ventes_dirty_{N_ROWS}.xlsx"

In [20]:
regions = ["Tunis", "Ariana", "Sfax", "Sousse", "Monastir", "Nabeul", "Bizerte", "Gabes"]

modes_paiement = ["Especes", "Carte Bancaire", "D17"]

start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 5, 31)

catalogue_list = [
    # Electronique
    {"categorie": "Electronique", "sous_categorie": "Telephones", "prix_min": 350,  "prix_max": 3500},
    {"categorie": "Electronique", "sous_categorie": "Ordinateurs Portables", "prix_min": 1200, "prix_max": 6000},
    {"categorie": "Electronique", "sous_categorie": "Tablettes", "prix_min": 400, "prix_max": 2200},
    {"categorie": "Electronique", "sous_categorie": "Ecouteurs", "prix_min": 25, "prix_max": 450},
    {"categorie": "Electronique", "sous_categorie": "Accessoires Telephone", "prix_min": 5, "prix_max": 90},

    # Mode et Vetements
    {"categorie": "Mode et Vetements", "sous_categorie": "Jeans", "prix_min": 45, "prix_max": 180},
    {"categorie": "Mode et Vetements", "sous_categorie": "Hoodies", "prix_min": 35, "prix_max": 130},
    {"categorie": "Mode et Vetements", "sous_categorie": "Chaussures", "prix_min": 60, "prix_max": 350},
    {"categorie": "Mode et Vetements", "sous_categorie": "Robes", "prix_min": 40, "prix_max": 200},
    {"categorie": "Mode et Vetements", "sous_categorie": "Vestes", "prix_min": 80, "prix_max": 280},

    # Beaute et Cosmetique
    {"categorie": "Beaute et Cosmetique", "sous_categorie": "Shampoing", "prix_min": 8, "prix_max": 45},
    {"categorie": "Beaute et Cosmetique", "sous_categorie": "Parfum", "prix_min": 35, "prix_max": 250},
    {"categorie": "Beaute et Cosmetique", "sous_categorie": "Skincare", "prix_min": 20, "prix_max": 130},
    {"categorie": "Beaute et Cosmetique", "sous_categorie": "Maquillage", "prix_min": 12, "prix_max": 90},
    {"categorie": "Beaute et Cosmetique", "sous_categorie": "Cremes", "prix_min": 10, "prix_max": 60},

    # Maison et Cuisine
    {"categorie": "Maison et Cuisine", "sous_categorie": "Lampes", "prix_min": 15, "prix_max": 120},
    {"categorie": "Maison et Cuisine", "sous_categorie": "Oreillers", "prix_min": 20, "prix_max": 80},
    {"categorie": "Maison et Cuisine", "sous_categorie": "Decoration", "prix_min": 10, "prix_max": 150},
    {"categorie": "Maison et Cuisine", "sous_categorie": "Ustensiles", "prix_min": 8, "prix_max": 95},
    {"categorie": "Maison et Cuisine", "sous_categorie": "Mixeurs", "prix_min": 90, "prix_max": 550},

    # Sports et Loisirs
    {"categorie": "Sports et Loisirs", "sous_categorie": "Equipement Fitness", "prix_min": 80, "prix_max": 900},
    {"categorie": "Sports et Loisirs", "sous_categorie": "Velos", "prix_min": 350, "prix_max": 2500},
    {"categorie": "Sports et Loisirs", "sous_categorie": "Chaussures Sport", "prix_min": 60, "prix_max": 280},
    {"categorie": "Sports et Loisirs", "sous_categorie": "Raquettes", "prix_min": 30, "prix_max": 200},
    {"categorie": "Sports et Loisirs", "sous_categorie": "Balles", "prix_min": 5, "prix_max": 40},
]

df_catalogue = pd.DataFrame(catalogue_list)
print(f"Catalogue: {len(df_catalogue)} sub-categories across {df_catalogue['categorie'].nunique()} categories")
print(df_catalogue.to_string(index=False))

Catalogue: 25 sub-categories across 5 categories
           categorie        sous_categorie  prix_min  prix_max
        Electronique            Telephones       350      3500
        Electronique Ordinateurs Portables      1200      6000
        Electronique             Tablettes       400      2200
        Electronique             Ecouteurs        25       450
        Electronique Accessoires Telephone         5        90
   Mode et Vetements                 Jeans        45       180
   Mode et Vetements               Hoodies        35       130
   Mode et Vetements            Chaussures        60       350
   Mode et Vetements                 Robes        40       200
   Mode et Vetements                Vestes        80       280
Beaute et Cosmetique             Shampoing         8        45
Beaute et Cosmetique                Parfum        35       250
Beaute et Cosmetique              Skincare        20       130
Beaute et Cosmetique            Maquillage        12        90
Beaute

In [21]:
def inject_missing_values(mode, cat, sous_cat, prix):
    if random.random() < 0.04: mode     = np.nan
    if random.random() < 0.04: cat      = np.nan; sous_cat = np.nan
    if random.random() < 0.03: prix     = np.nan
    return mode, cat, sous_cat, prix


def inject_duplicate_id(i, base_id):   
    if i > 10 and random.random() < 0.03:
        return f"VNT-{random.randint(1, i - 1):04d}"
    return base_id


def inject_date_format(date):
    r = random.random()
    if r < 0.05:   return date.strftime("%d/%m/%Y")
    elif r < 0.08: return date.strftime("%Y/%m/%d")
    return date.strftime("%Y-%m-%d")


def inject_region_typo(region):
    if random.random() < 0.06:
        return random.choice(["tunis", "SFAX", "ariana", "Sousse ", " Monastir", "nabeul"])
    return region


def inject_bad_quantity(qte):
    if random.random() < 0.02:
        return random.choice([0, -1, -random.randint(1, 5)])
    if random.random() < 0.02:
        return str(qte) + ".0"
    return qte


def inject_outlier_price(prix, prix_max):
    if pd.notna(prix) and random.random() < 0.02:
        return round(random.uniform(prix_max * 4, prix_max * 8), 2)
    return prix


def inject_category_typo(cat):
    bad = ["Electronique", "Mode & Vetements", "beaute et cosmetique", "maison cuisine"]
    if pd.notna(cat) and random.random() < 0.03:
        return random.choice(bad)
    return cat


print("Dirty injection functions defined.")


Dirty injection functions defined.


In [22]:
random.seed(SEED)
np.random.seed(SEED)

rows = []

for i in range(1, N_ROWS + 1):
    entry = df_catalogue.sample(1).iloc[0]
    cat = entry["categorie"]
    sous_cat = entry["sous_categorie"]
    p_min = entry["prix_min"]
    p_max = entry["prix_max"]

    prix = round(random.uniform(p_min, p_max), 2)
    qte = random.randint(1, 10)
    date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
    mode = random.choice(modes_paiement)
    region = random.choice(regions)
    sale_id = f"VNT-{i:04d}"

    mode, cat, sous_cat, prix = inject_missing_values(mode, cat, sous_cat, prix)
    sale_id = inject_duplicate_id(i, sale_id)
    date_str = inject_date_format(date)
    region = inject_region_typo(region)
    qte = inject_bad_quantity(qte)
    prix = inject_outlier_price(prix, p_max)
    cat = inject_category_typo(cat)

    rows.append({
        "ID_Vente": sale_id,
        "Date_Vente": date_str,
        "Region": region,
        "Categorie_Produit": cat,
        "Sous_categorie": sous_cat,
        "Prix_Unitaire": prix,
        "Quantite": qte,
        "Mode_Paiement": mode,
    })

df = pd.DataFrame(rows)
print(f"Generated {len(df)} rows.")
df.head(10)


Generated 200 rows.


,ID_Vente,Date_Vente,Region,Categorie_Produit,Sous_categorie,Prix_Unitaire,Quantite,Mode_Paiement
0,VNT-0001,2026-01-29,Sousse,Mode et Vetements,Robes,142.31,1,Carte Bancaire
1,VNT-0002,03/01/2026,Bizerte,Beaute et Cosmetique,Maquillage,14.07,4,D17
2,VNT-0003,2025-01-24,Nabeul,Maison et Cuisine,Lampes,50.34,2,Especes
3,VNT-0004,2025/08/14,Ariana,Sports et Loisirs,Balles,26.65,6,Especes
4,VNT-0005,2025-11-17,Ariana,Maison et Cuisine,Ustensiles,40.21,4,Carte Bancaire
5,VNT-0006,2024-08-22,Nabeul,Beaute et Cosmetique,Shampoing,39.19,1,Especes
6,VNT-0007,2024-09-09,Monastir,Mode et Vetements,Chaussures,101.43,3,D17
7,VNT-0008,2026-03-22,Bizerte,Maison et Cuisine,Oreillers,29.17,3,D17
8,VNT-0009,2026-02-26,Nabeul,Maison et Cuisine,Ustensiles,84.98,5,D17
9,VNT-0010,2026-05-11,Sousse,Mode et Vetements,Vestes,2160.59,-1,D17


In [23]:
valid_regions = {"Tunis", "Ariana", "Sfax", "Sousse", "Monastir", "Nabeul", "Bizerte", "Gabes"}
valid_cats = set(df_catalogue["categorie"].unique())

print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")

print(df.isnull().sum())

dupes = df[df.duplicated(subset="ID_Vente", keep=False)]
print(f"Rows with duplicate IDs: {len(dupes)}")
if len(dupes):
    print(dupes[["ID_Vente", "Date_Vente", "Region"]].head(6))

print(f"  dd/mm/yyyy : {df['Date_Vente'].str.match(r'^\\d{2}/').sum()}")
print(f"  yyyy/mm/dd : {df['Date_Vente'].str.match(r'^\\d{4}/').sum()}")
print(f"  yyyy-mm-dd : {df['Date_Vente'].str.match(r'^\\d{4}-').sum()}")

bad = ~df["Region"].apply(lambda x: str(x).strip() in valid_regions)
print(f"Bad region rows: {bad.sum()}")
if bad.sum():
    print(df.loc[bad, "Region"].value_counts())

num_qte = pd.to_numeric(df["Quantite"], errors="coerce")
print(f"  Negative or zero : {(num_qte <= 0).sum()}")
print(f"  Stored as string : {df['Quantite'].apply(lambda x: isinstance(x, str)).sum()}")

num_prix = pd.to_numeric(df["Prix_Unitaire"], errors="coerce")
print(f"  Prices above 5000 TND : {(num_prix > 5000).sum()}")
if (num_prix > 5000).sum():
    print(df.loc[num_prix > 5000, ["ID_Vente", "Sous_categorie", "Prix_Unitaire"]])

bad_cat = df["Categorie_Produit"].notna() & ~df["Categorie_Produit"].isin(valid_cats)
print(f"Bad category rows: {bad_cat.sum()}")
if bad_cat.sum():
    print(df.loc[bad_cat, "Categorie_Produit"].value_counts())

clean = df[df["Prix_Unitaire"].notna() & df["Sous_categorie"].notna()].copy()
clean["Prix_Unitaire"] = pd.to_numeric(clean["Prix_Unitaire"], errors="coerce")
print(clean.groupby("Sous_categorie")["Prix_Unitaire"].agg(["min", "mean", "max"]).round(1).to_string())

Rows: 200  |  Columns: 8
ID_Vente              0
Date_Vente            0
Region                0
Categorie_Produit     6
Sous_categorie        6
Prix_Unitaire         2
Quantite              0
Mode_Paiement        10
dtype: int64
Rows with duplicate IDs: 8
    ID_Vente  Date_Vente    Region
9   VNT-0010  2026-05-11    Sousse
22  VNT-0023  2025-07-12    Nabeul
34  VNT-0023  2025-02-02     Tunis
35  VNT-0036  2025-05-14  Monastir
40  VNT-0041  2026-03-26     Tunis
84  VNT-0010  2025-12-21    Nabeul
  dd/mm/yyyy : 0
  yyyy/mm/dd : 0
  yyyy-mm-dd : 0
Bad region rows: 9
Region
ariana    5
nabeul    2
SFAX      1
tunis     1
Name: count, dtype: int64
  Negative or zero : 2
  Stored as string : 7
  Prices above 5000 TND : 2
     ID_Vente         Sous_categorie  Prix_Unitaire
14   VNT-0015  Ordinateurs Portables        5336.03
176  VNT-0177  Ordinateurs Portables        5771.51
Bad category rows: 2
Categorie_Produit
beaute et cosmetique    2
Name: count, dtype: int64
                          

In [30]:
OUTPUT_PATH = "ventes.xlsx"

try:
    df.to_excel(OUTPUT_PATH, index=False)
    print(f"Saved {len(df)} rows to: {OUTPUT_PATH}")

except ImportError:
    OUTPUT_PATH = "ventes.csv"
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {len(df)} rows to: {OUTPUT_PATH}")


Saved 200 rows to: ventes.csv


# **Nettoyage du dataset**

**Chargement & inspection**


In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [2]:

df = pd.read_csv('ventes.csv')
print(df.shape)
df.head(10)

(200, 8)


,ID_Vente,Date_Vente,Region,Categorie_Produit,Sous_categorie,Prix_Unitaire,Quantite,Mode_Paiement
0,VNT-0001,2026-01-29,Sousse,Mode et Vetements,Robes,142.31,1.0,Carte Bancaire
1,VNT-0002,03/01/2026,Bizerte,Beaute et Cosmetique,Maquillage,14.07,4.0,D17
2,VNT-0003,2025-01-24,Nabeul,Maison et Cuisine,Lampes,50.34,2.0,Especes
3,VNT-0004,2025/08/14,Ariana,Sports et Loisirs,Balles,26.65,6.0,Especes
4,VNT-0005,2025-11-17,Ariana,Maison et Cuisine,Ustensiles,40.21,4.0,Carte Bancaire
5,VNT-0006,2024-08-22,Nabeul,Beaute et Cosmetique,Shampoing,39.19,1.0,Especes
6,VNT-0007,2024-09-09,Monastir,Mode et Vetements,Chaussures,101.43,3.0,D17
7,VNT-0008,2026-03-22,Bizerte,Maison et Cuisine,Oreillers,29.17,3.0,D17
8,VNT-0009,2026-02-26,Nabeul,Maison et Cuisine,Ustensiles,84.98,5.0,D17
9,VNT-0010,2026-05-11,Sousse,Mode et Vetements,Vestes,2160.59,-1.0,D17


**Info & valeurs manquantes**

In [3]:
df.info()
print('\nNaN par colonne:')
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID_Vente           200 non-null    str    
 1   Date_Vente         200 non-null    str    
 2   Region             200 non-null    str    
 3   Categorie_Produit  194 non-null    str    
 4   Sous_categorie     194 non-null    str    
 5   Prix_Unitaire      198 non-null    float64
 6   Quantite           200 non-null    float64
 7   Mode_Paiement      190 non-null    str    
dtypes: float64(2), str(6)
memory usage: 12.6 KB

NaN par colonne:
ID_Vente              0
Date_Vente            0
Region                0
Categorie_Produit     6
Sous_categorie        6
Prix_Unitaire         2
Quantite              0
Mode_Paiement        10
dtype: int64


**Suppression des doublons sur ID_Vente**

In [4]:
before = len(df)
dupes = df[df.duplicated(subset='ID_Vente', keep=False)].copy()

print(f"IDs dupliques detectes : {df.duplicated(subset='ID_Vente').sum()}")
if len(dupes):
    print("Lignes concernees :")
    display(dupes[['ID_Vente','Date_Vente','Categorie_Produit','Prix_Unitaire']])

df = df.drop_duplicates(subset='ID_Vente', keep='first').reset_index(drop=True)
print(f"\nLignes supprimees : {before - len(df)}")
print(f"Lignes restantes   : {len(df)}")

IDs dupliques detectes : 4
Lignes concernees :


,ID_Vente,Date_Vente,Categorie_Produit,Prix_Unitaire
9,VNT-0010,2026-05-11,Mode et Vetements,2160.59
22,VNT-0023,2025-07-12,Maison et Cuisine,26.71
34,VNT-0023,2025-02-02,Maison et Cuisine,38.96
35,VNT-0036,2025-05-14,NaN,107.99
40,VNT-0041,2026-03-26,Maison et Cuisine,404.76
84,VNT-0010,2025-12-21,Beaute et Cosmetique,34.21
123,VNT-0041,2024-05-31,Beaute et Cosmetique,48.73
189,VNT-0036,2025-09-15,Beaute et Cosmetique,59.94



Lignes supprimees : 4
Lignes restantes   : 196


**Standardisation des formats de date**

In [5]:
def parse_date(val):
    """Detecte et convertit n'importe lequel des trois formats en yyyy-mm-dd."""
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    val = str(val).strip()
    if re.match(r'^\d{2}/\d{2}/\d{4}$', val):
        return datetime.strptime(val, '%d/%m/%Y').strftime('%Y-%m-%d')
    if re.match(r'^\d{4}/\d{2}/\d{2}$', val):
        return datetime.strptime(val, '%Y/%m/%d').strftime('%Y-%m-%d')
    if re.match(r'^\d{4}-\d{2}-\d{2}$', val):
        return val
    return np.nan

formats_avant = {
    'dd/mm/yyyy': df['Date_Vente'].str.match(r'^\d{2}/').sum(),
    'yyyy/mm/dd': df['Date_Vente'].str.match(r'^\d{4}/').sum(),
    'yyyy-mm-dd': df['Date_Vente'].str.match(r'^\d{4}-').sum(),
}
print("Formats avant nettoyage :", formats_avant)

df['Date_Vente'] = df['Date_Vente'].apply(parse_date)

formats_apres = {
    'dd/mm/yyyy': df['Date_Vente'].str.match(r'^\d{2}/').sum(),
    'yyyy/mm/dd': df['Date_Vente'].str.match(r'^\d{4}/').sum(),
    'yyyy-mm-dd': df['Date_Vente'].str.match(r'^\d{4}-').sum(),
}
print("Formats apres nettoyage :", formats_apres)
print(f"Dates non parsees (NaN) : {df['Date_Vente'].isna().sum()}")

Formats avant nettoyage : {'dd/mm/yyyy': np.int64(6), 'yyyy/mm/dd': np.int64(7), 'yyyy-mm-dd': np.int64(183)}
Formats apres nettoyage : {'dd/mm/yyyy': np.int64(0), 'yyyy/mm/dd': np.int64(0), 'yyyy-mm-dd': np.int64(196)}
Dates non parsees (NaN) : 0


**Normalisation de la colonne Region**

In [7]:
VALID_REGIONS = {
    'Tunis', 'Ariana', 'Sfax', 'Sousse',
    'Monastir', 'Nabeul', 'Bizerte', 'Gabes'
}

def clean_region(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().title()
    return val if val in VALID_REGIONS else np.nan

print("Valeurs uniques avant nettoyage :")
print(sorted(df['Region'].dropna().unique()))

df['Region'] = df['Region'].apply(clean_region)

print("\nValeurs uniques apres nettoyage :")
print(sorted(df['Region'].dropna().unique()))
print(f"Regions invalides converties en NaN : {df['Region'].isna().sum()}")


Valeurs uniques avant nettoyage :
[' Monastir', 'Ariana', 'Bizerte', 'Gabes', 'Monastir', 'Nabeul', 'SFAX', 'Sfax', 'Sousse', 'Sousse ', 'Tunis', 'ariana', 'nabeul', 'tunis']

Valeurs uniques apres nettoyage :
['Ariana', 'Bizerte', 'Gabes', 'Monastir', 'Nabeul', 'Sfax', 'Sousse', 'Tunis']
Regions invalides converties en NaN : 0


**Normalisation de Categorie_Produit**

In [8]:
VALID_CATS = [
    'Electronique',
    'Mode et Vetements',
    'Beaute et Cosmetique',
    'Maison et Cuisine',
    'Sports et Loisirs',
]

CAT_KEYWORDS = {
    'Electronique':        ['electron'],
    'Mode et Vetements':   ['mode', 'vet', 'vete'],
    'Beaute et Cosmetique':['beau', 'cosm'],
    'Maison et Cuisine':   ['maison', 'cuisi'],
    'Sports et Loisirs':   ['sport', 'lois'],
}

def normalize_categorie(val):
    if pd.isna(val):
        return np.nan
    val_lower = str(val).strip().lower()
    for canonical, keywords in CAT_KEYWORDS.items():
        if any(kw in val_lower for kw in keywords):
            return canonical
    return np.nan

print("Valeurs uniques avant nettoyage :")
print(df['Categorie_Produit'].value_counts(dropna=False).to_string())

df['Categorie_Produit'] = df['Categorie_Produit'].apply(normalize_categorie)

print("\nValeurs uniques apres nettoyage :")
print(df['Categorie_Produit'].value_counts(dropna=False).to_string())

Valeurs uniques avant nettoyage :
Categorie_Produit
Maison et Cuisine       54
Sports et Loisirs       41
Beaute et Cosmetique    39
Mode et Vetements       27
Electronique            27
NaN                      6
beaute et cosmetique     2

Valeurs uniques apres nettoyage :
Categorie_Produit
Maison et Cuisine       54
Beaute et Cosmetique    41
Sports et Loisirs       41
Mode et Vetements       27
Electronique            27
NaN                      6


**Nettoyage de la colonne Quantite**

In [9]:
print("Exemples de valeurs brutes :")
print(df["Quantite"].unique()[:20])

df_clean = df.copy()

df_clean["Quantite"] = pd.to_numeric(df_clean["Quantite"], errors="coerce")

df_clean["Quantite"] = df_clean["Quantite"].apply(
    lambda x: int(x) if pd.notna(x) and x == int(x) else x
)

invalid_qty = df_clean["Quantite"] <= 0
print(f"\nQuantites <= 0 detectees : {invalid_qty.sum()}")
print(df_clean.loc[invalid_qty, ["ID_Vente", "Quantite"]])

df_clean.loc[invalid_qty, "Quantite"] = np.nan

mediane_qte = int(round(df_clean["Quantite"].median()))
df_clean["Quantite"] = df_clean["Quantite"].fillna(mediane_qte).astype(int)

print(f"\nMediane utilisee pour imputation : {mediane_qte}")
print(f"NaN restants dans Quantite : {df_clean['Quantite'].isna().sum()}")

Exemples de valeurs brutes :
[ 1.  4.  2.  6.  3.  5. -1. 10.  8.  9.  7.  0.]

Quantites <= 0 detectees : 2
     ID_Vente  Quantite
9    VNT-0010        -1
178  VNT-0182         0

Mediane utilisee pour imputation : 5
NaN restants dans Quantite : 0


**Nettoyage de Prix_Unitaire**

In [10]:
PRIX_MAX_REF = {
    'Telephones': 3500, 'Ordinateurs Portables': 6000, 'Tablettes': 2200,
    'Ecouteurs': 450, 'Accessoires Telephone': 90,
    'Jeans': 180, 'Hoodies': 130, 'Chaussures': 350, 'Robes': 200, 'Vestes': 280,
    'Shampoing': 45, 'Parfum': 250, 'Skincare': 130, 'Maquillage': 90, 'Cremes': 60,
    'Lampes': 120, 'Oreillers': 80, 'Decoration': 150, 'Ustensiles': 95, 'Mixeurs': 550,
    'Equipement Fitness': 900, 'Velos': 2500, 'Chaussures Sport': 280,
    'Raquettes': 200, 'Balles': 40,
}
PRIX_MIN_REF = {
    'Telephones': 350, 'Ordinateurs Portables': 1200, 'Tablettes': 400,
    'Ecouteurs': 25, 'Accessoires Telephone': 5,
    'Jeans': 45, 'Hoodies': 35, 'Chaussures': 60, 'Robes': 40, 'Vestes': 80,
    'Shampoing': 8, 'Parfum': 35, 'Skincare': 20, 'Maquillage': 12, 'Cremes': 10,
    'Lampes': 15, 'Oreillers': 20, 'Decoration': 10, 'Ustensiles': 8, 'Mixeurs': 90,
    'Equipement Fitness': 80, 'Velos': 350, 'Chaussures Sport': 60,
    'Raquettes': 30, 'Balles': 5,
}

df_clean['Prix_Unitaire'] = pd.to_numeric(df_clean['Prix_Unitaire'], errors='coerce')

def flag_outlier_prix(row):
    sc = row['Sous_categorie']
    px = row['Prix_Unitaire']
    if pd.isna(px) or pd.isna(sc) or sc not in PRIX_MAX_REF:
        return False
    return px > PRIX_MAX_REF[sc] * 3

outliers_mask = df_clean.apply(flag_outlier_prix, axis=1)
print(f"Outliers de prix detectés : {outliers_mask.sum()}")

if outliers_mask.any():
    display(df_clean.loc[outliers_mask, ['ID_Vente', 'Sous_categorie', 'Prix_Unitaire']])

df_clean.loc[outliers_mask, 'Prix_Unitaire'] = np.nan
print(f"\nNaN dans Prix_Unitaire après invalidation : {df_clean['Prix_Unitaire'].isna().sum()}")

mediane_par_sous_cat = df_clean.groupby('Sous_categorie')['Prix_Unitaire'].transform('median')
mediane_globale = df_clean['Prix_Unitaire'].median()
df_clean['Prix_Unitaire'] = (
    df_clean['Prix_Unitaire']
    .fillna(mediane_par_sous_cat)
    .fillna(mediane_globale)
    .round(2)
)
print(f"NaN restants dans Prix_Unitaire : {df_clean['Prix_Unitaire'].isna().sum()}")

Outliers de prix detectés : 4


,ID_Vente,Sous_categorie,Prix_Unitaire
9,VNT-0010,Vestes,2160.59
129,VNT-0133,Chaussures,2315.16
133,VNT-0137,Hoodies,633.31
150,VNT-0154,Parfum,1207.16



NaN dans Prix_Unitaire après invalidation : 6
NaN restants dans Prix_Unitaire : 0


**Traitement des valeurs manquantes restantes**

In [12]:
SOUS_CAT_TO_CAT = {
    'Telephones': 'Electronique', 'Ordinateurs Portables': 'Electronique',
    'Tablettes': 'Electronique', 'Ecouteurs': 'Electronique',
    'Accessoires Telephone': 'Electronique',
    'Jeans': 'Mode et Vetements', 'Hoodies': 'Mode et Vetements',
    'Chaussures': 'Mode et Vetements', 'Robes': 'Mode et Vetements',
    'Vestes': 'Mode et Vetements',
    'Shampoing': 'Beaute et Cosmetique', 'Parfum': 'Beaute et Cosmetique',
    'Skincare': 'Beaute et Cosmetique', 'Maquillage': 'Beaute et Cosmetique',
    'Cremes': 'Beaute et Cosmetique',
    'Lampes': 'Maison et Cuisine', 'Oreillers': 'Maison et Cuisine',
    'Decoration': 'Maison et Cuisine', 'Ustensiles': 'Maison et Cuisine',
    'Mixeurs': 'Maison et Cuisine',
    'Equipement Fitness': 'Sports et Loisirs', 'Velos': 'Sports et Loisirs',
    'Chaussures Sport': 'Sports et Loisirs', 'Raquettes': 'Sports et Loisirs',
    'Balles': 'Sports et Loisirs',
}

mask_cat_nan = df_clean['Categorie_Produit'].isna() & df_clean['Sous_categorie'].notna()
df_clean.loc[mask_cat_nan, 'Categorie_Produit'] = (
    df_clean.loc[mask_cat_nan, 'Sous_categorie'].map(SOUS_CAT_TO_CAT)
)

mode_paiement = df_clean['Mode_Paiement'].mode(dropna=True)
mode_paiement = mode_paiement.iloc[0] if len(mode_paiement) else np.nan
df_clean['Mode_Paiement'] = df_clean['Mode_Paiement'].fillna(mode_paiement)

print(f"Categorie_Produit NaN apres deduction : {df_clean['Categorie_Produit'].isna().sum()}")
print(f"Mode_Paiement impute avec '{mode_paiement}' : NaN restants = {df_clean['Mode_Paiement'].isna().sum()}")

print("\nNaN restants par colonne :")
print(df_clean.isnull().sum())

Categorie_Produit NaN apres deduction : 6
Mode_Paiement impute avec 'Especes' : NaN restants = 0

NaN restants par colonne :
ID_Vente             0
Date_Vente           0
Region               0
Categorie_Produit    6
Sous_categorie       6
Prix_Unitaire        0
Quantite             0
Mode_Paiement        0
dtype: int64


**Typage final des colonnes**

In [13]:
df_clean['Date_Vente'] = pd.to_datetime(df_clean['Date_Vente'], errors='coerce')
df_clean['Prix_Unitaire'] = df_clean['Prix_Unitaire'].astype(float)
df_clean['Quantite'] = df_clean['Quantite'].astype(int)

df = df_clean.copy()

print(df.dtypes)
print("\nApercu final :")
display(df.head(10))

ID_Vente                        str
Date_Vente           datetime64[us]
Region                          str
Categorie_Produit               str
Sous_categorie                  str
Prix_Unitaire               float64
Quantite                      int64
Mode_Paiement                   str
dtype: object

Apercu final :


,ID_Vente,Date_Vente,Region,Categorie_Produit,Sous_categorie,Prix_Unitaire,Quantite,Mode_Paiement
0,VNT-0001,2026-01-29,Sousse,Mode et Vetements,Robes,142.31,1,Carte Bancaire
1,VNT-0002,2026-01-03,Bizerte,Beaute et Cosmetique,Maquillage,14.07,4,D17
2,VNT-0003,2025-01-24,Nabeul,Maison et Cuisine,Lampes,50.34,2,Especes
3,VNT-0004,2025-08-14,Ariana,Sports et Loisirs,Balles,26.65,6,Especes
4,VNT-0005,2025-11-17,Ariana,Maison et Cuisine,Ustensiles,40.21,4,Carte Bancaire
5,VNT-0006,2024-08-22,Nabeul,Beaute et Cosmetique,Shampoing,39.19,1,Especes
6,VNT-0007,2024-09-09,Monastir,Mode et Vetements,Chaussures,101.43,3,D17
7,VNT-0008,2026-03-22,Bizerte,Maison et Cuisine,Oreillers,29.17,3,D17
8,VNT-0009,2026-02-26,Nabeul,Maison et Cuisine,Ustensiles,84.98,5,D17
9,VNT-0010,2026-05-11,Sousse,Mode et Vetements,Vestes,194.90,5,D17


**Export du dataset propre**

In [15]:
OUTPUT_PATH = "ventes_propre.xlsx"

try:
    df.to_excel(OUTPUT_PATH, index=False)
    print(f"Dataset propre sauvegarde : {OUTPUT_PATH}")
except ImportError:
    OUTPUT_PATH = "ventes_propre.csv"
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Dataset propre sauvegarde : {OUTPUT_PATH}")

print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")

Dataset propre sauvegarde : ventes_propre.csv
Dimensions : 196 lignes x 8 colonnes
